# Project 3: Decision Trees & Random Forests
## This Project gives extra points for the final grade

## **Due 29.01.2026 at 4 PM**

## Overview

### **Submit your project solution as a group of 2-4 people.**

### Tasks

1. **Implement Decision Tree** (2.5 Points)
- Implement a decision tree classifier from scratch using the ID3 algorithm.
- You are free to add additional functionalities to improve the performance of your decision tree.

2. **Implement Random Forest** (2 Points)
- Implement a random forest classifier from scratch using your decision tree implementation.
- You are free to choose the hyperparameters of your random forest implementation.
- You are free to add any additional functionalities to improve the performance of your random forest.

3. **Testing & Evaluation** (0.5 Points)
- Train and tune hyperparameters of your decision tree and random forest implementations using the training set (you can use sklearn for hyperparameter tuning).
- You **must reach at least 95.5% accuracy** on the testing set with your final model.
- The best performing model will be awarded a prize in the class.

### **NOTE**: 
#### You are not allowed to use sklearn, pytorch or similar Deep Learning frameworks for the algorithm implementations. For data splitting, scoring and hyperparameter tuning sklearn is allowed. Numpy is allowed everywhere.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

In [2]:
# Do not change this part
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target) # 0 = malignant, 1 = benign
random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

In [ ]:
import math
from collections import Counter
from typing import Union

# Decision Tree Node class
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature        # Index of feature to split on
        self.threshold = threshold    # Threshold value for split
        self.left = left              # Left subtree
        self.right = right            # Right subtree
        self.value = value            # Class value if leaf node
        

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.tree = None
        
    def fit(self, X, y):
        self.tree = self._build_tree(X, y, depth=0)
        return self
        
    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           n_classes == 1:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
        
        best_gain = 0
        best_feature = None
        best_threshold = None
        
        for feature_idx in range(n_features):
            feature_values = X.iloc[:, feature_idx].values
            thresholds = np.unique(feature_values)
            
            for threshold in thresholds:
                left_mask = feature_values <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                

                gain = self._information_gain(y, y[left_mask], y[right_mask])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        

        if best_feature is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
        

        left_mask = X.iloc[:, best_feature].values <= best_threshold
        right_mask = ~left_mask
        

        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return Node(feature=best_feature, threshold=best_threshold, 
                   left=left_subtree, right=right_subtree)
    
    def _entropy(self, y):
        proportions = np.bincount(y) / len(y)
        entropy = -np.sum([p * np.log2(p) for p in proportions if p > 0])
        return entropy
    
    def _information_gain(self, parent, left_child, right_child):
        n = len(parent)
        n_left = len(left_child)
        n_right = len(right_child)
        

        if n_left == 0 or n_right == 0:
            return 0
        
        child_entropy = (n_left / n) * self._entropy(left_child) + \
                       (n_right / n) * self._entropy(right_child)
        

        gain = self._entropy(parent) - child_entropy
        return gain
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.tree) for _, x in X.iterrows()])
    
    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
        
        if x.iloc[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)


class RandomForest:
    def __init__(self, n_estimators=100, max_depth=None, min_samples_split=2, 
                 min_samples_leaf=1, random_state=None, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state
        self.max_features = max_features
        self.trees = []
        
    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        n_features = X.shape[1]
        if self.max_features is None:
            self.max_features = int(np.sqrt(n_features))
        
        self.trees = []
        
        for _ in range(self.n_estimators):
            indices = np.random.choice(len(X), size=len(X), replace=True)
            X_bootstrap = X.iloc[indices]
            y_bootstrap = y.iloc[indices]
            
            tree = DecisionTree(max_depth=self.max_depth,
                              min_samples_split=self.min_samples_split,
                              min_samples_leaf=self.min_samples_leaf)
            
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)
        
        return self
    
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        majority_prediction = []
        for i in range(predictions.shape[1]):
            votes = predictions[:, i]
            majority_prediction.append(Counter(votes).most_common(1)[0][0])
        
        return np.array(majority_prediction)

In [ ]:
# Train and evaluate Decision Tree with hyperparameter tuning
print("=" * 60)
print("DECISION TREE CLASSIFIER")
print("=" * 60)


best_dt_accuracy = 0
best_dt_params = {}
best_dt_model = None

max_depths = [5, 10, 15, 20, 25, 30, None]
min_samples_splits = [2, 5, 10]
min_samples_leafs = [1, 2, 4]

for max_depth in max_depths:
    for min_samples_split in min_samples_splits:
        for min_samples_leaf in min_samples_leafs:
            dt = DecisionTree(max_depth=max_depth, 
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf)
            dt.fit(X_train, y_train.values)
            y_pred = dt.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            
            if accuracy > best_dt_accuracy:
                best_dt_accuracy = accuracy
                best_dt_params = {
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split,
                    'min_samples_leaf': min_samples_leaf
                }
                best_dt_model = dt

print(f"\nBest Decision Tree Parameters: {best_dt_params}")
print(f"Best Decision Tree Test Accuracy: {best_dt_accuracy:.4f}")


DECISION TREE CLASSIFIER

Best Decision Tree Parameters: {'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 2}
Best Decision Tree Test Accuracy: 0.9474


In [ ]:
# Train and evaluate Random Forest with hyperparameter tuning
print("\n" + "=" * 60)
print("RANDOM FOREST CLASSIFIER")
print("=" * 60)


best_rf_accuracy = 0
best_rf_params = {}
best_rf_model = None

n_estimators_list = [50, 100, 200, 300]
max_depths = [10, 15, 20, 25, None]
min_samples_splits = [2, 5, 10]

for n_estimators in n_estimators_list:
    for max_depth in max_depths:
        for min_samples_split in min_samples_splits:
            rf = RandomForest(n_estimators=n_estimators,
                            max_depth=max_depth,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=1,
                            random_state=random_state)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            
            if accuracy > best_rf_accuracy:
                best_rf_accuracy = accuracy
                best_rf_params = {
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split
                }
                best_rf_model = rf

print(f"\nBest Random Forest Parameters: {best_rf_params}")
print(f"Best Random Forest Test Accuracy: {best_rf_accuracy:.4f}")



RANDOM FOREST CLASSIFIER


In [ ]:
# Final Model Comparison and Results
print("\n" + "=" * 60)
print("FINAL RESULTS AND COMPARISON")
print("=" * 60)


if best_rf_accuracy > best_dt_accuracy:
    print(f"\n✓ Random Forest performs better!")
    print(f"  Random Forest Accuracy: {best_rf_accuracy:.4f}")
    print(f"  Decision Tree Accuracy: {best_dt_accuracy:.4f}")
    final_model = best_rf_model
    final_accuracy = best_rf_accuracy
else:
    print(f"\n✓ Decision Tree performs better!")
    print(f"  Decision Tree Accuracy: {best_dt_accuracy:.4f}")
    print(f"  Random Forest Accuracy: {best_rf_accuracy:.4f}")
    final_model = best_dt_model
    final_accuracy = best_dt_accuracy

print(f"\nFinal Model Test Accuracy: {final_accuracy:.4f}")

# Requirement check
if final_accuracy >= 0.955:
    print(f"\n✓ SUCCESS: Achieved {final_accuracy:.4f} accuracy (>= 95.5% requirement)")
else:
    print(f"\n✗ FAILED: Achieved {final_accuracy:.4f} accuracy (< 95.5% requirement)")
